In [1]:
import time
from typing import Dict, Any

In [2]:
class AHE:
    def __init__(self, q: int, m: int, n: int, w:int):
        self.q = q
        self.m = m
        self.n = n
        self.w = w
        
        self.F = GF(q)
        self.Fext = GF(q^m, 'z')
        
        Tmp.<y> = PolynomialRing(self.F)
        Q = Tmp.random_element(degree=self.n)
        while not Q.is_irreducible():
            Q = Tmp.random_element(degree=self.n)
        
        self.PR = PolynomialRing(self.Fext, 'x')
        self.x = self.PR.gen()
        self.Q = self.PR(Q)

    @staticmethod
    def vec(a):
        return vector(list(a))

    def _mat(self, a):
        return matrix(self.F, [list(elem) for elem in a]).T

    def _rank_norm(self, a):
        return rank(self._mat(a))

    def _find_B(self, f):
        B = self._mat(f)
        cand_cols = identity_matrix(self.F, self.m)
        for i in range(self.m):
            tmp = B.augment(cand_cols[:,i])
            if rank(tmp) == rank(B) + 1:
                B = tmp
            if B.ncols() == self.m:
                break
    
        return B

    def _sample_with_supp(self, f):
        c = random_vector(self.F, len(f))
    
        return f.inner_product(c)

    def _poly(self, v):
        return self.PR(list(v))

    def _poly_to_vec(self, p, d):
        p_list = list(p)
        
        return vector(self.Fext, p_list + (d+1-len(p_list))*[0])

    def _vec_vec_prod(self, u, v):
        u_poly = self._poly(u)
        v_poly = self._poly(v)
        tmp = (u_poly * v_poly) % self.Q
    
        return self._poly_to_vec(tmp, len(u)-1)

    def _get_f(self): 
        f = random_vector(self.Fext, self.w)
        while self._rank_norm(f) != self.w:
            f = random_vector(self.Fext,self.w)
    
        return f

    def _get_g(self, B):
        z = self.Fext.gen() 
    
        z_pows = vector(self.Fext,[z**i for i in range(self.m)])
        g = z_pows * B[:,self.w:]

        return g

    def key_gen(self):
        f = self._get_f()
        B = self._find_B(f)
        g = self._get_g(B)
        D = (B**(-1)).T[:,self.w:]
        s = vector(self.Fext, [self._sample_with_supp(f) for _ in range(self.n)])

        return f,g,D,s

    def encrypt(self, message, key):
        f,g,D,s = key
        
        u = random_vector(self.Fext, self.n)
        p = self._poly(u)
        R2 = random_matrix(self.F,self.w,self.n)
        e = f*R2
        v = self._vec_vec_prod(s,u) + e + (message*g[0])
        ct = (u,v)
    
        return ct

    def decrypt(self, ct, key):
        f,g,D,s = key
        
        u,v = ct
        d = D.column(0)

        tmp = v - self._vec_vec_prod(s, u)
    
        message = [d.inner_product(self.vec(tmp[j])) for j in range(len(tmp))]
    
        return vector(message)


    @staticmethod
    def sum(ct1, ct2):
        return (ct1[0] + ct2[0], ct1[1] + ct2[1])

    def mul_const(self, ct, msg):
        u = self._vec_vec_prod(ct[0], msg)
        v = self._vec_vec_prod(ct[1], msg)

        return (u, v)

In [3]:
ahe = AHE(2, 15, 11, 3)
sk = ahe.key_gen()

msg1 = random_vector(ahe.F, ahe.n)
msg2 = random_vector(ahe.F, ahe.n)

ct1 = ahe.encrypt(msg1, sk)
ct2 = ahe.encrypt(msg2, sk)

ct3 = ahe.sum(ct1,ct2)

print(ahe.decrypt(ct3,sk) == msg1+msg2)

True


In [4]:
class SHE(AHE):
    def _get_g(self, B, d_F):
        z = self.Fext.gen() 
    
        z_pows = vector(self.Fext,[z**i for i in range(self.m)])
        g = z_pows * B[:,d_F:]
    
        return g

    def key_gen(self):
        f = self._get_f()
        
        rw = 0
        d_F = 1
        while d_F + 2!= rw:
            g1 = self.Fext.random_element()
            fg1 = f*g1
        
            gens = (
                list(f) +
                list(fg1) +
                [f[i] * f[j] for i in range(self.w) for j in range(i, self.w)]
            )
            
            M = matrix(self.F, [self.vec(g) for g in gens])
        
            F_ = M.row_space().basis()
            g2 = g1*g1
        
            f_ = vector(self.Fext, [self.Fext(list(b)) for b in F_])
            check = vector(self.Fext, list(f_) + [g1, g2])
    
            d_F = len(f_)
            rw = self._rank_norm(check)
    
        B = self._find_B(check)
        g = self._get_g(B,d_F)
        D = (B**(-1)).T[:,d_F:]
        s = vector(self.Fext, [self._sample_with_supp(f) for _ in range(self.n)])
    
        return f,g,D,s

    def mul_ct(self, ct1, ct2):
        u1,v1 = ct1
        u2,v2 = ct2
    
        a = self._vec_vec_prod(v1,v2)
        b = -(self._vec_vec_prod(u1,v2) + self._vec_vec_prod(u2,v1))
        c = self._vec_vec_prod(u1,u2)
    
        return a,b,c

    def decrypt_mul(self, ct, key):
        f,g,D,s = key
        a,b,c = ct
    
        sb = self._vec_vec_prod(s,b)
        ss = self._vec_vec_prod(s,s)
        s2c = self._vec_vec_prod(ss,c)
    
        tmp = a + sb + s2c
    
        d = D[:,1]
    
        message = (d).T * self._mat(tmp)
    
        return vector(message)

In [5]:
she = SHE(2, 15, 11, 3)
sk = she.key_gen()

m1 = random_vector(she.F, she.n)
m2 = random_vector(she.F, she.n)

ct1 = she.encrypt(m1, sk)
ct2 = she.encrypt(m2, sk)

ct_mul = she.mul_ct(ct1, ct2)
result = she.decrypt_mul(ct_mul, sk)
expected = she._vec_vec_prod(m1, m2)

print(result == expected)

True


In [6]:
params = [
    (2, 172, 20, 13),
    (2, 367, 183, 7),
    (2, 1296, 314, 6),
    (2, 3125, 713, 5)
]

In [21]:
for i in range(len(params)):
    print(f'Params: q={params[i][0]} m={params[i][1]} n={params[i][2]} w={params[i][3]}')
    she = SHE(*params[i])
    start = time.time()
    sk = she.key_gen()
    end = time.time()
    print(f'key gen time: {end - start}')

    msg1 = random_vector(she.F, she.n)
    msg2 = random_vector(she.F, she.n)

    start = time.time()
    ct1 = she.encrypt(msg1, sk)
    end = time.time()
    ct2 = she.encrypt(msg2, sk)

    print(f'enc time: {end-start}')
    
    start = time.time()
    res = she.sum(ct1, ct2)
    end = time.time()

    print(f'time sum: {end-start}')

    start = time.time()
    decrypt_sum = she.decrypt(res, sk)
    end = time.time()

    print(f'res sum: {decrypt_sum == msg1+msg2}\ndecrypt sum time: {end-start}')    

    start = time.time()
    res = she.mul_ct(ct1, ct2)
    end = time.time()

    print(f'time mul: {end-start}')

    start = time.time()
    decrypt_mul = she.decrypt_mul(res, sk)
    end = time.time()

    print(f'res mul: {decrypt_mul == she._vec_vec_prod(msg1, msg2)}\ndecrypt mul time: {end - start}\n===============\n')

Params: q=2 m=172 n=20 w=13
key gen time: 0.17865276336669922
enc time: 0.0368499755859375
time sum: 5.7220458984375e-06
res sum: True
decrypt sum time: 0.030102014541625977
time mul: 0.08591675758361816
res mul: True
decrypt mul time: 0.06861615180969238

Params: q=2 m=367 n=183 w=7
key gen time: 0.3342170715332031
enc time: 0.7330489158630371
time sum: 4.7206878662109375e-05
res sum: True
decrypt sum time: 0.6172280311584473
time mul: 1.795450210571289
res mul: True
decrypt mul time: 1.4356811046600342

Params: q=2 m=1296 n=314 w=6
key gen time: 3.238663911819458
enc time: 4.478367328643799
time sum: 9.322166442871094e-05
res sum: True
decrypt sum time: 3.782517910003662
time mul: 11.068338871002197
res mul: True
decrypt mul time: 8.731934785842896

Params: q=2 m=3125 n=713 w=5
key gen time: 25.403653144836426
enc time: 25.992872953414917
time sum: 0.00028896331787109375
res sum: True
decrypt sum time: 21.602844953536987
time mul: 65.62310791015625
res mul: True
decrypt mul time: 50.

In [7]:
she = SHE(*params[0])
sk = she.key_gen()

In [8]:
class ObviousPIRClient:
    def __init__(self, she, sk):
        self.she = she
        self.sk = sk

    def query(self, i, N):
        vectors = [zero_vector(self.she.F, self.she.n) for _ in range(N)]
        vectors[i][0] = 1

        ct_vectors = []
        for v in vectors:
            ct = self.she.encrypt(v, self.sk)
            ct_vectors.append(ct)

        return ct_vectors

    def decode(self, ct):
        return self.she.decrypt(ct, self.sk)

In [9]:
class ObviousPIRServer:
    def __init__(self, she):
        self.she = she

    def answer(self, q, db):
        res = None
        for i in range(db.nrows()):
            if res is None:
                res = she.mul_const(q[i],db[i])
            else:
                res = she.sum(res, she.mul_const(q[i],db[i]))

        return res

In [10]:
db_len = 100
idx = 3
db = random_matrix(she.F,db_len,she.n)
print(db[idx])

client = ObviousPIRClient(she, sk)
q = client.query(idx, db_len)

server = ObviousPIRServer(she)
resp = server.answer(q, db)

res = client.decode(resp)
print(res == db[idx])

(1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0)
True


In [11]:
 class PIRClient:
    def __init__(self, she, sk):
         self.she = she
         self.sk = sk

    def query(self, i, num_cols):
        vectors = [zero_vector(self.she.F, self.she.n) for _ in range(num_cols)]
        vectors[i][0] = 1

        ct_vectors = []
        for v in vectors:
            ct = self.she.encrypt(v, self.sk)
            ct_vectors.append(ct)

        return ct_vectors

    def decode_сolumn(self, ct):
        res = []
        for i in range(len(ct)):
            res.append(self.she.decrypt(ct[i], self.sk))
        
        return res

    def decode(self, ct, pos, sqrt_n):
        return self.she.decrypt(ct[pos // sqrt_n], self.sk)

In [12]:
class PIRServer:
    def __init__(self, she):
        self.she = she

    def answer(self, q, db):
        sqrt_n = ceil(sqrt(db.nrows()))
        results = []
        for r in range(sqrt_n):                    
            res = None
            for j in range(sqrt_n):                
                if res is None:
                    res = she.mul_const(q[j], db[r * sqrt_n + j])
                else:
                    res = she.sum(res, she.mul_const(q[j], db[r * sqrt_n + j]))
            results.append(res)

        return results

In [13]:
# пересобрать матрицу если запрос – 2мб, а хранится 10кб
db_len = 100
idx = 3
db = random_matrix(she.F,db_len,she.n)
print(db[idx])

client = PIRClient(she, sk)

q = client.query(idx%sqrt(db_len), db_len)

server = PIRServer(she)
resp = server.answer(q,db)

print(client.decode_сolumn(resp))
res = client.decode(resp, idx, sqrt(db_len))
print(res == db[idx])

(1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1)
[(1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1), (1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1), (0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0), (0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0), (0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0), (0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1), (1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0), (0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1), (0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0), (1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1)]
True


In [15]:
class TensorPIRClient:
    def __init__(self, she, sk):
        self.she = she
        self.sk = sk

    def query(self, idx, N):
        sqrt_n = int(ceil(sqrt(N)))
        row = idx // sqrt_n
        col = idx % sqrt_n

        q_row = []
        for i in range(sqrt_n):
            v = zero_vector(self.she.F, self.she.n)
            if i == row:
                v[0] = 1
            q_row.append(self.she.encrypt(v, self.sk))

        q_col = []
        for j in range(sqrt_n):
            v = zero_vector(self.she.F, self.she.n)
            if j == col:
                v[0] = 1
            q_col.append(self.she.encrypt(v, self.sk))

        return q_row, q_col

    def decode(self, ct):
        return self.she.decrypt_mul(ct, self.sk)

In [18]:
class TensorPIRServer:
    def __init__(self, she):
        self.she = she

    @staticmethod
    def sum_triple(ct1, ct2):
        return (ct1[0] + ct2[0], ct1[1] + ct2[1], ct1[2] + ct2[2])

    def answer(self, query, db):
        q_row, q_col = query
        sqrt_n = int(ceil(sqrt(db.nrows())))

        col_results = []
        for i in range(sqrt_n):
            res = None
            for j in range(sqrt_n):
                db_idx = i * sqrt_n + j
                if db_idx < db.nrows():
                    term = self.she.mul_const(q_col[j], db[db_idx])
                    if res is None:
                        res = term
                    else:
                        res = self.she.sum(res, term)
            col_results.append(res)

        final = None
        for i in range(sqrt_n):
            if col_results[i] is None:
                continue
            term = self.she.mul_ct(q_row[i], col_results[i])
            if final is None:
                final = term
            else:
                final = self.sum_triple(final, term)

        return final

In [19]:
db_len = 100
idx = 3
db = random_matrix(she.F, db_len, she.n)
print(f"Target: {db[idx]}")

client = TensorPIRClient(she, sk)
query = client.query(idx, db_len)

server = TensorPIRServer(she)
resp = server.answer(query, db)

res = client.decode(resp)
print(f"Result: {res}")
print(f"Match: {res == db[idx]}")

Target: (0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0)
Result: (0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0)
Match: True
